In [1]:
!pip install lightgbm


[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import glob
import os
import random
import sys

import numpy as np
import pandas as pd

In [3]:
try:
  from google.colab import drive
  IS_GOOGLE_COLAB = True
except ImportError:
  IS_GOOGLE_COLAB = False

if IS_GOOGLE_COLAB:
  mount_path='/content/drive'
  base_folder = "/Data/FX"
  data_folder = f'{mount_path}/My Drive{base_folder}'
  drive.mount(mount_path)
else:
  base_folder = 'L:/data'
  data_folder = os.path.join(base_folder, "fx")

In [4]:
ohlc_column = ['open','high','low','close']
file_name = f"OANDA-Japan MT5 Live/mt5_USDJPY_TickToMIN1.csv"

file_path = os.path.join(data_folder, file_name)
file_path = os.path.abspath(file_path)
df_1m = pd.read_csv(file_path, parse_dates=True, index_col=0)
df_1m.tail()

,open,high,low,close,spread,volume
time,,,,,,
2022-08-31 17:55:00,138.538,138.587,138.532,138.567,0.008,0.0
2022-08-31 17:56:00,138.567,138.574,138.501,138.540,0.008,0.0
2022-08-31 17:57:00,138.540,138.592,138.540,138.592,0.006,0.0
2022-08-31 17:58:00,138.591,138.643,138.581,138.625,0.009,0.0
2022-08-31 17:59:00,138.625,138.643,138.596,138.623,0.005,0.0


In [5]:
import finance_client

### MACD Cross

#### Check Expected Output

In [6]:
process = finance_client.fprocess.MACDProcess(short_window=12, long_window=26, signal_window=9,
                                              target_column='close')

In [7]:
df = process(df_1m)
df.tail()

,open,high,low,close,spread,volume,macd_S_EMA,macd_L_EMA,MACD,macd_Signal
time,,,,,,,,,,
2022-08-31 17:55:00,138.538,138.587,138.532,138.567,0.008,0.0,138.566715,138.607671,-0.040956,-0.046686
2022-08-31 17:56:00,138.567,138.574,138.501,138.540,0.008,0.0,138.562605,138.602658,-0.040053,-0.045359
2022-08-31 17:57:00,138.540,138.592,138.540,138.592,0.006,0.0,138.567127,138.601869,-0.034742,-0.043236
2022-08-31 17:58:00,138.591,138.643,138.581,138.625,0.009,0.0,138.576031,138.603582,-0.027552,-0.040099
2022-08-31 17:59:00,138.625,138.643,138.596,138.623,0.005,0.0,138.583257,138.605020,-0.021764,-0.036432


In [8]:
signals = df["MACD"] > df["macd_Signal"]
df["MACDValue"] = df["MACD"] - df["macd_Signal"]
df["PositionSignal"] = signals.astype(int).diff()
df.tail()

,open,high,low,close,spread,volume,macd_S_EMA,macd_L_EMA,MACD,macd_Signal,MACDValue,PositionSignal
time,,,,,,,,,,,,
2022-08-31 17:55:00,138.538,138.587,138.532,138.567,0.008,0.0,138.566715,138.607671,-0.040956,-0.046686,0.005729,0.0
2022-08-31 17:56:00,138.567,138.574,138.501,138.540,0.008,0.0,138.562605,138.602658,-0.040053,-0.045359,0.005306,0.0
2022-08-31 17:57:00,138.540,138.592,138.540,138.592,0.006,0.0,138.567127,138.601869,-0.034742,-0.043236,0.008494,0.0
2022-08-31 17:58:00,138.591,138.643,138.581,138.625,0.009,0.0,138.576031,138.603582,-0.027552,-0.040099,0.012547,0.0
2022-08-31 17:59:00,138.625,138.643,138.596,138.623,0.005,0.0,138.583257,138.605020,-0.021764,-0.036432,0.014668,0.0


In [9]:
# クローズ価格
price = df["close"]

# クローズが発生する行（±1）
exit_points = df.index[df["PositionSignal"].abs() == 1]

# 各行に「次のクローズ行」を割り当てる
# next_exit_idx を numpy array として扱う
next_idx = np.searchsorted(exit_points, df.index, side="right")

# 範囲外は -1 にする
next_idx = np.where(next_idx < len(exit_points), next_idx, -1)

# exit index（-1 は NaN に）
df["next_exit_idx"] = pd.to_datetime(
    np.where(next_idx >= 0, exit_points[next_idx], np.datetime64("NaT"))
)

# 次のクローズ価格
df["exit_price"] = price.reindex(df["next_exit_idx"]).values

# 現在ポジション方向（1=Long, -1=Short）
df["position"] = signals.replace({True: 1, False: -1})

# 損益計算
df["pnl_to_next_signal"] = (df["exit_price"] - price) * df["position"]


df.head()

C:\Users\ksato\AppData\Local\Temp\ipykernel_30128\3948727450.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["position"] = signals.replace({True: 1, False: -1})


,open,high,low,close,spread,volume,macd_S_EMA,macd_L_EMA,MACD,macd_Signal,MACDValue,PositionSignal,next_exit_idx,exit_price,position,pnl_to_next_signal
time,,,,,,,,,,,,,,,,
2016-08-31 18:00:00,103.486,103.490,103.424,103.426,0.008,0.0,103.426000,103.426000,0.000000,0.000000,0.000000,NaN,2016-08-31 18:08:00,103.443,-1,-0.017
2016-08-31 18:01:00,103.426,103.435,103.410,103.422,0.008,0.0,103.425385,103.425704,-0.000319,-0.000064,-0.000255,0.0,2016-08-31 18:08:00,103.443,-1,-0.021
2016-08-31 18:02:00,103.422,103.447,103.404,103.414,0.008,0.0,103.423633,103.424837,-0.001204,-0.000292,-0.000912,0.0,2016-08-31 18:08:00,103.443,-1,-0.029
2016-08-31 18:03:00,103.413,103.416,103.379,103.395,0.007,0.0,103.419228,103.422627,-0.003399,-0.000913,-0.002485,0.0,2016-08-31 18:08:00,103.443,-1,-0.048
2016-08-31 18:04:00,103.395,103.398,103.359,103.387,0.006,0.0,103.414270,103.419988,-0.005718,-0.001874,-0.003844,0.0,2016-08-31 18:08:00,103.443,-1,-0.056


In [10]:
def make_position(df, tf):
    return np.where(
        df[f"MACD_{tf}"] > df[f"Signal_{tf}"],
        1,
        -1
    )

def make_exit_signal(df, tf):
    sig = df[f"MACD_{tf}"] > df[f"Signal_{tf}"]
    return sig.astype(int).diff()

def calc_pnl_1m(df, position, exit_signal, price_col="close"):
    price = df[price_col]

    # 反転した時刻
    exit_points = df.index[exit_signal.abs() == 1]

    next_idx = np.searchsorted(exit_points, df.index, side="right")
    next_idx = np.where(next_idx < len(exit_points), next_idx, -1)

    next_exit_time = pd.to_datetime(
        np.where(next_idx >= 0, exit_points[next_idx], np.datetime64("NaT"))
    )

    exit_price = price.reindex(next_exit_time).values

    pnl = (exit_price - price) * position
    return pnl

In [11]:
timeframes = ["5min", "15min", "30min", "1h", "4h", "8h", "1D"]

for tf in timeframes:
    df_tf = (
        df_1m
        .resample(tf)
        .agg({
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum"
        })
        .dropna()
    )

    # MACD計算
    df_tf = process(df_tf)

    # 1分足に forward-fill で展開
    df[f"MACD_{tf}"] = df_tf["MACD"].reindex(df.index, method="ffill")
    df[f"Signal_{tf}"] = df_tf["macd_Signal"].reindex(df.index, method="ffill")
    df[f"MACDValue_{tf}"] = df[f"MACD_{tf}"] - df[f"Signal_{tf}"]
    ps = make_position(df, tf)
    exit_signal = make_exit_signal(df, tf)
    df[f"pnl_{tf}"] = calc_pnl_1m(df, ps, exit_signal)

#### LiteGBM

In [26]:
features = [
    "MACDValue", "MACDValue_5min", "MACDValue_15min",
    "MACDValue_30min", "MACDValue_1h", "MACDValue_4h", "MACDValue_8h", "MACDValue_1D"
]

targets = ["pnl_to_next_signal", "pnl_5min", "pnl_15min",
           "pnl_30min", "pnl_1h", "pnl_4h", "pnl_8h", "pnl_1D"]

df_ml = df[features + targets].iloc[480:].dropna()

In [27]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

X = df_ml[features]
models = []

for target in targets:
    y = df_ml[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, shuffle=False, test_size=0.2
    )

    model = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)
    models.append(model)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006579 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 1778639, number of used features: 8
[LightGBM] [Info] Start training from score -0.001044
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

#### State Space Model

##### HMM

In [28]:
!pip install hmmlearn

     -------------------------------------- 125.9/125.9 kB 1.2 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
from hmmlearn.hmm import GaussianHMM
import numpy as np

features = [
    "MACDValue", "MACDValue_5min", "MACDValue_15min",
    "MACDValue_30min", "MACDValue_1h"
]

X = df[features].dropna().values
K = 6  # 状態数

model = GaussianHMM(
    n_components=K,
    covariance_type="full",
    n_iter=200,
    random_state=42
)

model.fit(X)

states = model.predict(X)

df_hmm = df.loc[df[features].dropna().index].copy()
df_hmm["state"] = states


In [ ]:
# 各状態ごとの平均損益を表示

for s in range(K):
    print(f"\nState {s}")
    print(
        df_hmm[df_hmm["state"] == s][
            ["pnl_to_next_signal", "pnl_5min", "pnl_15min", "pnl_30min", "pnl_1h"]
        ].mean()
    )


State 0
pnl_to_next_signal   -0.001338
pnl_5min              0.013476
pnl_15min             0.026321
pnl_30min             0.051913
pnl_1h                0.096276
dtype: float64

State 1
pnl_to_next_signal   -0.000977
pnl_5min              0.016035
pnl_15min             0.037816
pnl_30min             0.062954
pnl_1h                0.102340
dtype: float64

State 2
pnl_to_next_signal   -0.001468
pnl_5min              0.014288
pnl_15min             0.027664
pnl_30min             0.046607
pnl_1h                0.074456
dtype: float64

State 3
pnl_to_next_signal   -0.001278
pnl_5min              0.013624
pnl_15min             0.026441
pnl_30min             0.051925
pnl_1h                0.096302
dtype: float64

State 4
pnl_to_next_signal    0.000151
pnl_5min              0.028660
pnl_15min             0.049677
pnl_30min             0.080499
pnl_1h                0.128834
dtype: float64

State 5
pnl_to_next_signal   -0.000814
pnl_5min              0.016743
pnl_15min             0.034753
pnl